# 01. What is in this file, and what has already been done to it?

Data: PBMC scRNA-seq, 18 RA vs 18 matched controls, JCI Insight 178499, CELLxGENE `3b751975`. Stored in `C:/bio_data/`, never in the repo.

In [2]:
import scanpy as sc
import pandas as pd

In [3]:
adata = sc.read_h5ad("C:/bio_data/ra_pbmc.h5ad")

reading the data

In [4]:
adata

AnnData object with n_obs × n_vars = 108717 × 21648
    obs: 'nCount_RNA', 'nFeature_RNA', 'Lane', 'demux_doublet_call', 'demux_RD_TOTL', 'demux_RD_PASS', 'demux_RD_UNIQ', 'demux_N_SNP', 'demux_PRB_DBL', 'percent_mt', 'scrub_doublets', 'batch', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'total_counts_ribo', 'pct_counts_ribo', 'total_counts_hb', 'pct_counts_hb', 'n_genes', 'n_counts', 'norm_library_size', 'S_score', 'G2M_score', 'phase', 'cell_cycle_diff', 'pair_index_CW', 'live_cells', 'dead_cells', 'pct_alive', 'MY_MTX', 'MY_bDMARD', 'MY_pt_global', 'MY_md_global', 'MY_tjc', 'MY_sjc', 'MY_esr', 'MY_crp', 'MY_cdai', 'MY_das28esr4', 'MY_das28crp4', 'MY_RF_status', 'MY_CCP_Status', 'activity_python_crp', 'activity_python_binary_crp', 'activity_python_esr', 'activity_python_binary_esr', 'leiden_r3.0', 'rough_annot', 'fine_annot', 'tissue_ontology_term_id', 'tissue_type', 'assay_ontology_term_id', 'disease_ontology_term_id', 'self_reported_ethnicity_ontology_t

Seeing what is in obs: a lot of columns including disease and donor_id

In [5]:
adata.obs.groupby("disease")["donor_id"].nunique()

disease
rheumatoid arthritis    18
normal                  18
Name: donor_id, dtype: int64

Checking if there are as many cases as controls. The numbers match the paper. 

In [6]:
adata.X.data

array([4.3039665, 5.680073 , 4.3039665, ..., 1.9891268, 1.9891268,
       1.9891268], shape=(28870411,), dtype=float32)

The data is log-normalized as we see fractions instead of raw counts. 

---

# Part 2: what has already been done to this file?

Your cells above show what is *in* the file. Before running any pipeline, find out what the authors already did to it. Every later step depends on these answers: you cannot QC data that is already filtered, or normalize data that is already normalized, without knowing it.

> **How this notebook works.** Each step has a **spec** (what the code must do), a **read-first** link, an empty code cell for you, and a **TEST** cell written by Claude. Your cell is done when its test prints PASS and you can say why your approach works. The ✍️ cells are yours to write; predictions go in *before* you run the cell they predict.
>
> No function names are given on purpose. Find them in the read-first link or the scanpy docs. Stuck for more than 10 minutes on one cell: ask Claude for a *question*, not code.

## E1. Where are the raw counts?

**Spec:** your cell above shows `.X` holds decimals. CELLxGENE requires raw counts to be stored somewhere in the file. Find them and store that matrix in `raw_counts`. Then check two things on its nonzero values: are they all whole numbers, and what is the largest one?

**Read first:** the AnnData docs, the attributes list at the top: https://anndata.readthedocs.io/en/stable/generated/anndata.AnnData.html (look for something that is not `.X` and not `.layers`).

✍️ **Prediction: where are they, and is .X then counts or something derived?**

_X is the matrix made from the data (obs,vars). It gives the total amount of observations_

In [7]:
# raw_counts = the integer count matrix, same cells and genes as adata
# then print: are all nonzero values whole numbers? what is the max?

raw_counts = adata.raw.X
print("Are all nonzero values whole numbers?", (raw_counts.data % 1 == 0).all())
print("Max value:", raw_counts.data.max())

Are all nonzero values whole numbers? True
Max value: 578.0


In [8]:
# TEST: do not edit. Run it after your cell; it must print PASS.
import numpy as np
assert raw_counts.shape == adata.shape, f"shape {raw_counts.shape} != {adata.shape}"
d = raw_counts.data[:500_000]
assert np.allclose(d, np.round(d)), "these are not whole numbers, so not raw counts"
print("PASS: raw counts found,", raw_counts.shape)

PASS: raw counts found, (108717, 21648)


## E2. Prove the log-normalization claim

Your markdown above says `.X` is log-normalized *because we see fractions*. Fractions alone do not prove that: a matrix divided by library size also has fractions, and is not logged.

**Spec, part 1:** standard scanpy normalization scales every cell to the same total, then takes `log(1 + x)`. The obvious check: undo the log on the first 1,000 cells and sum each row. Store the sums in `row_sums`.

**Read first:** https://www.sc-best-practices.org/preprocessing_visualization/normalization.html (the shifted logarithm section).

✍️ **Prediction: what will row_sums look like if the claim is true?**

_..._

In [9]:
# undo log1p on adata.X for the first 1000 cells, sum per cell -> row_sums

first = adata.X[:1000]
row_sums = np.asarray(np.expm1(first).sum(axis=1)).ravel()

In [10]:
# TEST: do not edit. Run it after your cell; it must print PASS.
assert len(row_sums) == 1000
print("PASS: computed. Now look at them: min", round(float(min(row_sums))), "max", round(float(max(row_sums))))

PASS: computed. Now look at them: min 2747 max 9583


✍️ **They are not all the same. Does that kill the claim? Do not answer yet; part 2 decides.**

_..._

**Spec, part 2: a sharper test.** If a cell was scaled by one factor, then for every gene `expm1(X) / raw` is the same number in that cell. For each of the first 1,000 cells: take the genes that are nonzero in both, compute that ratio, check its spread, then multiply the ratio by the cell's **raw** total. Store the median of that product in `target_sum_guess`.

In [12]:
i = 0
x = adata.X[i].toarray().ravel()        # log-normalized values, one cell
r = raw_counts[i].toarray().ravel()     # raw counts, same cell
shared = (x > 0) & (r > 0)              # genes nonzero in both
ratio = np.expm1(x[shared]) / r[shared]
print("ratio min", ratio.min(), "max", ratio.max(), "std", ratio.std())
print("ratio x raw total:", np.median(ratio) * r.sum())

ratio min 72.99268 max 72.99271 std 8.495735e-06
ratio x raw total: 10000.001


In [14]:
products = []
for i in range(1000):
    x = adata.X[i].toarray().ravel()
    r = raw_counts[i].toarray().ravel()
    shared = (x > 0) & (r > 0)
    ratio = np.expm1(x[shared]) / r[shared]
    products.append(np.median(ratio) * r.sum())

target_sum_guess = np.median(products)

In [15]:
# TEST: do not edit. Run it after your cell; it must print PASS.
import numpy as np
assert np.isclose(target_sum_guess, 10_000, rtol=0.02), f"got {target_sum_guess}"
print("PASS: every cell was scaled to", round(float(target_sum_guess)), "then log1p'd")

PASS: every cell was scaled to 10000 then log1p'd


**Spec, part 3: the mystery.** If every cell was scaled to 10,000, why did part 1 give smaller sums? Some genes have raw counts but are zero everywhere in `.X`. Find them: store how many in `n_removed_genes` and look at their names.

In [33]:
# genes that are nonzero somewhere in raw but zero in every cell of X
# n_removed_genes, and print 20 of their symbols
in_raw = (raw_counts > 0).sum(axis=0).A1 > 0    # gene seen in at least 1 cell of raw
in_X = (adata.X > 0).sum(axis=0).A1 > 0         # gene seen in at least 1 cell of X
removed = in_raw & ~in_X                         # in raw, but in no cell of X
n_removed_genes = removed.sum()
print("n_removed_genes", n_removed_genes)
print(adata.var.loc[removed, "feature_name"].head(20).tolist())


n_removed_genes 192
['MRPL20', 'RPL22', 'RPL11', 'RPS6KA1', 'MRPS15', 'RPS8', 'MRPL37', 'RPL5', 'MRPS21', 'MRPL9', 'RPS27', 'MRPL24', 'MRPS14', 'RPS6KC1', 'MRPL55', 'RPS7', 'MRPL33', 'RPS27A', 'MRPL53', 'MRPL19']


In [34]:
# TEST: do not edit. Run it after your cell; it must print PASS.
assert n_removed_genes == 192, n_removed_genes
print("PASS: 192 genes were removed from X after normalizing")

PASS: 192 genes were removed from X after normalizing


✍️ **Which kinds of genes did the authors remove, why would anyone do that, and which var column already flagged them? Then rewrite your claim from the top as a proven fact.**

_..._

## E3. Gene names

**Spec:** look at `adata.var_names`. Marker genes like CD14 are gene *symbols*, but the index here is something else. Find the `var` column that holds the symbols and store its *name* (a string) in `symbol_column`.

**Gotcha you will hit later:** a few symbols are not unique. Count how many symbols appear more than once and store it in `n_duplicate_symbols`.

In [ ]:
# symbol_column = name of the var column with gene symbols
# n_duplicate_symbols = how many symbols occur more than once

...

In [ ]:
# TEST: do not edit. Run it after your cell; it must print PASS.
syms = set(adata.var[symbol_column].astype(str))
for g in ["CD14", "MS4A1", "CD3E", "NKG7"]:
    assert g in syms, f"{g} not in that column"
assert isinstance(n_duplicate_symbols, (int, np.integer)) and n_duplicate_symbols >= 0
print("PASS: symbols live in", symbol_column, "|", n_duplicate_symbols, "duplicated")

## E4. Has QC already been done?

**Spec:** `obs` already has QC columns: `n_genes_by_counts`, `total_counts`, `pct_counts_mt`. Show their distribution (min, median, max is enough as a table; a histogram per column is better). Store the three maxima in `max_genes`, `max_counts`, `max_pct_mt`.

**Read first:** https://www.sc-best-practices.org/preprocessing_visualization/quality_control.html, the first two sections.

✍️ **Prediction: did the authors already filter cells? What would the distribution look like if they did?**

_..._

In [ ]:
# describe or plot the three QC columns
# max_genes, max_counts, max_pct_mt = their maxima

...

In [ ]:
# TEST: do not edit. Run it after your cell; it must print PASS.
assert max_genes == adata.obs["n_genes_by_counts"].max()
assert max_counts == adata.obs["total_counts"].max()
assert max_pct_mt == adata.obs["pct_counts_mt"].max()
print("PASS:", max_genes, max_counts, round(float(max_pct_mt), 2))

✍️ **What do those maxima tell you? (Look at how round they are.) And the median cell has only a few hundred UMIs: what does that mean for how many genes you can see per cell?**

_..._

## E5. The authors' labels: the answer key

**Spec:** `fine_annot` and `rough_annot` are the authors' annotations. These are your known answer for notebooks 03 to 05: you will try to recover them without looking. Count cells per label for both, store the number of distinct fine labels in `n_fine`, and make a crosstab of fine against rough.

In [ ]:
# value counts of fine_annot and rough_annot
# n_fine = number of distinct fine labels
# crosstab fine vs rough

...

In [ ]:
# TEST: do not edit. Run it after your cell; it must print PASS.
assert n_fine == 18, f"{n_fine}: the paper reports 18 subsets"
print("PASS: 18 subsets, as in the paper")

✍️ **Two placements in the rough labels are surprising (look at the gamma-delta T cells and the dendritic cells). Which, and why might the authors have done it?**

_..._

## E6. Cells per donor

**Spec:** count cells per `donor_id`, store as a Series `cells_per_donor`. Also check whether `batch` is confounded with disease: crosstab donors (one row per donor) by batch and disease.

In [ ]:
# cells_per_donor = Series, index donor_id, value = number of cells
# crosstab: per batch, how many RA and how many control donors?

...

In [ ]:
# TEST: do not edit. Run it after your cell; it must print PASS.
assert len(cells_per_donor) == 36
assert cells_per_donor.sum() == adata.n_obs
print("PASS: min", cells_per_donor.min(), "max", cells_per_donor.max())

✍️ **If one donor has 3x more cells than another, and you compare RA vs control by pooling all cells, what goes wrong? (This is the n = 4 lesson from 04 in a new form.) And is batch a problem for the RA vs control comparison?**

_..._

## E7. Disease activity per donor

**Spec:** build `donors`, a DataFrame with **one row per donor** (index `donor_id`) and columns `disease`, `sex`, `MY_das28crp4`, `activity_python_binary_crp`. Count active donors (DAS28-CRP >= 3.2) into `n_active`, and RA donors with no DAS28-CRP into `n_ra_missing`.

In [ ]:
# donors = one row per donor with the four columns
# n_active = RA donors with DAS28-CRP >= 3.2
# n_ra_missing = RA donors without a DAS28-CRP value

...

In [ ]:
# TEST: do not edit. Run it after your cell; it must print PASS.
assert donors.shape[0] == 36 and donors.index.is_unique
assert n_active == 7, n_active
assert n_ra_missing == 2, n_ra_missing
print("PASS: 7 active, 9 low/remission, 2 unknown")

✍️ **Active vs low is 7 vs 9 donors. Before you ever test it: what does 04 tell you to expect from a comparison this small?**

_..._

## Teach-back

3 to 5 sentences in your own words: what this notebook did, what it found, and one thing that surprised you. No sentence means not learned.

_..._

## AI note

What did Claude write (the specs, the tests), what did you write, and where did you get stuck and ask for a hint?

_..._